<a href="https://colab.research.google.com/github/witsarut-big-data/128-356-Big-Data-for-test/blob/main/BigData_Week7_Streaming_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Big Data สัปดาห์ที่ 7: Streaming Analytics
## Apache Kafka & Spark Structured Streaming

**สัปดาห์ที่:** 7  

> 🎯 **เป้าหมายวันนี้:**
> 1. เข้าใจแนวคิด **Streaming Analytics** vs Batch Processing
> 2. รู้จักสถาปัตยกรรม **Apache Kafka** ทุกส่วนประกอบ
> 3. เขียน **Spark Structured Streaming** Queries จริง
> 4. ทำ **Lab ครบ** บน Google Colab ไม่ต้องติดตั้งอะไรเพิ่ม
> 5. เชื่อมโยงกับ Week 2 (Parquet), 4 (Spark), 5 (Pipeline), 6 (Airflow)

**แหล่งอ้างอิง:** [Spark Docs](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) | [Kafka Docs](https://kafka.apache.org/documentation/) | [Medium: Real-time Data with Spark+Kafka](https://medium.com/@yuvrender.karan/real-time-data-streaming-with-apache-kafka-and-spark-structured-streaming-edb2c47a36f7)

**ใช้ได้ทั้ง**: Google Colab ✅ / Local Jupyter ✅  

---


# 0. 🧠 ทบทวน — เชื่อมโยงกับสัปดาห์ก่อน

| สัปดาห์ | เนื้อหาหลัก | เชื่อมกับ Week 7 |
|---------|------------|------------------|
| **W1** | Big Data 5Vs | Streaming แก้ปัญหา **Velocity** (V ตัวที่ 2) |
| **W2** | Parquet, Columnar Storage | Streaming sink ผลลัพธ์ลงไฟล์ **Parquet** |
| **W3** | DuckDB, Spark SQL | ใช้ **SQL query** บน Streaming DataFrame |
| **W4** | Spark Architecture, DAG | Structured Streaming สร้างบน **Spark Engine** เดิม |
| **W5** | Data Pipeline ETL | Streaming = **Continuous Pipeline** ไม่มีจุดสิ้นสุด |
| **W6** | Airflow, DAG Orchestration | Airflow **trigger/monitor** Spark Streaming Jobs |
| **W8** | EDA & KPI | ใช้ข้อมูลจาก Streaming (Parquet) มาทำ **Dashboard** |

> 💡 **ภาพรวม:** Streaming Analytics คือ Data Pipeline ที่ `ไม่มีจุดสิ้นสุด` — ข้อมูลไหลเข้าตลอด 24/7 และเราวิเคราะห์ได้ทันที


# 1. ⚡ Batch vs Streaming — สองโลกของการประมวลผลข้อมูล


<div align="center">
  <img src="https://github.com/witsarutsarai12-Academic/128-356-Big-Data/blob/main/images/week7/batch_vs_streaming.png?raw=1" width="800" alt="Batch Processing vs Stream Processing — Latency, Data Type, Complexity">
  <br><i>Batch Processing vs Stream Processing — Latency, Data Type, Complexity</i>
</div>


## 1.1 Batch Processing

```
ข้อมูล D1..D1000 สะสมทั้งวัน
         │
    [00:00] ──────────── Spark Batch Job ──────────▶ Report
         └─ ประมวลผลครั้งเดียว ทุกคืนเที่ยงคืน
```
- **Use Case:** รายงานยอดขายรายวัน, Payroll รายเดือน, ML Training
- **Keyword:** `spark.read`, `df.write`, Airflow DAG ทำงานรายวัน

## 1.2 Stream Processing

```
D1 ──▶ Process ──▶ Result  (t = 0.1s)
D2 ──▶ Process ──▶ Result  (t = 0.2s)
D3 ──▶ Process ──▶ Result  (t = 0.3s)  ← ตลอดเวลา!
```
- **Use Case:** Fraud Detection, Live Dashboard, IoT, Stock Price Alert
- **Keyword:** `readStream`, `writeStream`, Kafka, Trigger

## 1.3 Lambda Architecture — ใช้ทั้งสองแบบพร้อมกัน

```
Raw Data
   ├──▶ [Speed Layer]  Spark Streaming ──▶ Real-time View  (วินาที)
   └──▶ [Batch Layer]  Spark Batch     ──▶ Batch View      (ชั่วโมง)
                               └─ Merge ──▶ Serving Layer (API/Dashboard)
```
> 💡 **Kappa Architecture (ทางเลือก):** simplify โดยใช้ Streaming อย่างเดียว + replay จาก Kafka เพื่อแทน Batch

| | **Lambda** | **Kappa** |
|---|---|---|
| **Batch Layer** | ✅ มี | ❌ ไม่มี |
| **Speed Layer** | ✅ มี | ✅ มี (Kafka replay) |
| **ความซับซ้อน** | สูง | ต่ำกว่า |
| **ใช้งานจริง** | Netflix, LinkedIn | Uber, Cloudflare |


# 2. 📨 Apache Kafka — Message Broker สำหรับ Streaming

> **Apache Kafka** คือ Distributed Event Streaming Platform ออกแบบโดย LinkedIn (2011) ปัจจุบัน Open Source บน Apache Foundation  
> **GitHub:** [apache/kafka](https://github.com/apache/kafka) ⭐ 29k+ stars


<div align="center">
  <img src="https://github.com/witsarutsarai12-Academic/128-356-Big-Data/blob/main/images/week7/kafka_architecture.png?raw=1" width="800" alt="Apache Kafka Architecture: Producers → Brokers (Topics/Partitions) → Consumer Groups">
  <br><i>Apache Kafka Architecture: Producers → Brokers (Topics/Partitions) → Consumer Groups</i>
</div>


## 2.1 ส่วนประกอบหลัก (Core Components)

| ส่วนประกอบ | ภาษาไทย | บทบาท | เปรียบเทียบ |
|-----------|---------|-------|------------|
| **Producer** | ผู้ส่งข้อมูล | เขียนข้อมูลเข้า Topic | นักข่าวส่งข่าว |
| **Topic** | หัวข้อ/ช่องทาง | หมวดหมู่ข้อมูล | ช่องทีวี |
| **Partition** | ส่วนแบ่ง | แบ่ง Topic เพื่อ Parallel | แผนกในบริษัท |
| **Offset** | ตำแหน่ง | Index ของ Message ใน Partition | เลขหน้าหนังสือ |
| **Broker** | เซิร์ฟเวอร์ | เก็บและส่ง Messages | ออฟฟิศไปรษณีย์ |
| **Consumer** | ผู้รับข้อมูล | อ่าน Messages จาก Topic | ผู้สมัครรับข่าวสาร |
| **Consumer Group** | กลุ่มผู้รับ | หลายตัวช่วยกันอ่าน Parallel | ทีมงานหลายคน |
| **ZooKeeper/KRaft** | ผู้จัดการ Cluster | จัดการ Metadata, Leader Election | CEO บริษัท |

## 2.2 Topic และ Partition

```
Topic: 'ecommerce-orders'  (Partitions = 3, Replication Factor = 2)

Partition 0: [ORD-001|t=0] [ORD-004|t=3] [ORD-007|t=6] ...
                offset=0      offset=1      offset=2

Partition 1: [ORD-002|t=1] [ORD-005|t=4] [ORD-008|t=7] ...

Partition 2: [ORD-003|t=2] [ORD-006|t=5] [ORD-009|t=8] ...
```
**Key Insight จาก GitHub Example ([polomarcus/Spark-Structured-Streaming-Examples](https://github.com/polomarcus/Spark-Structured-Streaming-Examples)):**
- ยิ่ง Partition มาก → ยิ่ง Parallel มาก → ยิ่งเร็ว
- Replication Factor = 2 หมายถึง ข้อมูลมีสำเนาใน 2 Broker
- Kafka เก็บข้อมูลถาวร (Retention 7 วัน default) สามารถ **Replay** ได้

## 2.3 Consumer Groups — อ่าน Parallel

```
Topic: orders (3 Partitions)

Consumer Group A (Spark App):        Consumer Group B (Analytics):
  Consumer 1 ← Partition 0             Consumer 1 ← Partition 0,1,2
  Consumer 2 ← Partition 1             (อ่านทั้งหมดคนเดียว)
  Consumer 3 ← Partition 2

💡 แต่ละ Group อ่านอิสระจากกัน — ไม่กระทบกัน!
```

## 2.4 Exactly-Once Semantics — ไม่สูญหาย ไม่ซ้ำ

| Delivery Mode | ความหมาย | ตัวอย่าง |
|---|---|---|
| **At-most-once** | ส่งได้สูงสุด 1 ครั้ง อาจสูญหาย | Log ที่ไม่สำคัญ |
| **At-least-once** | ส่งอย่างน้อย 1 ครั้ง อาจซ้ำ | Push Notification |
| **Exactly-once** | ส่งครั้งเดียวพอดี ไม่สูญ ไม่ซ้ำ | Financial Transactions |

> 💡 Spark Structured Streaming + Kafka รองรับ Exactly-once ด้วย `enable.idempotence=true` และ Checkpointing


# 3. 🔥 Spark Structured Streaming

> **Spark Structured Streaming** เป็น Stream Processing engine ที่สร้างบน Spark SQL  
> ใช้ DataFrame API เดิม — เขียนเหมือน Batch แต่ทำงานแบบ Streaming!


<div align="center">
  <img src="https://github.com/witsarutsarai12-Academic/128-356-Big-Data/blob/main/images/week7/spark_streaming_pipeline.png?raw=1" width="800" alt="Spark Structured Streaming Pipeline: readStream → Transform → writeStream → Sink">
  <br><i>Spark Structured Streaming Pipeline: readStream → Transform → writeStream → Sink</i>
</div>


## 3.1 DataFrame API เปรียบเทียบ Batch vs Stream

```python
# ── BATCH (Week 4-5) ──────────────────────────────────────────────
df = spark.read.csv('orders.csv', header=True)   # อ่านครั้งเดียว
result = df.groupBy('product').count()            # Transformation (Lazy)
result.show()                                     # Action (trigger)

# ── STREAMING (Week 7) ────────────────────────────────────────────
df = spark.readStream.format('socket')..load()   # อ่านต่อเนื่อง
result = df.groupBy('product').count()            # เหมือนกันทุกอย่าง!
result.writeStream.outputMode('complete')...start() # Trigger ต่อเนื่อง
```

## 3.2 Output Modes

| Mode | เมื่อไหร่ใช้ | ตัวอย่าง |
|------|------------|----------|
| **Append** | ข้อมูลใหม่ที่ไม่เปลี่ยนแปลง | Event log, กรณี filter/select |
| **Complete** | ต้องการผลรวมทั้งหมดทุก Trigger | `groupBy().count()` |
| **Update** | เฉพาะแถวที่เปลี่ยนแปลง | `groupBy()` แต่ไม่ต้องการทั้งหมด |

## 3.3 Trigger Types

```python
# Default: ทำทันทีที่ micro-batch ก่อนเสร็จ
.trigger(processingTime='0 seconds')

# Fixed interval: ทุก N วินาที
.trigger(processingTime='10 seconds')

# Once: ทำครั้งเดียวแล้วหยุด (คล้าย Batch)
.trigger(once=True)

# Continuous: latency ต่ำมาก (experimental)
.trigger(continuous='1 second')
```


# 4. 🕐 Window Aggregation & Watermark


<div align="center">
  <img src="https://github.com/witsarutsarai12-Academic/128-356-Big-Data/blob/main/images/week7/window_watermark.png?raw=1" width="800" alt="Window Aggregation: Tumbling Window (Non-overlapping) vs Sliding Window (Overlapping) + Watermark for Late Data">
  <br><i>Window Aggregation: Tumbling Window (Non-overlapping) vs Sliding Window (Overlapping) + Watermark for Late Data</i>
</div>


## 4.1 Tumbling Window — ไม่ซ้อนทับ

```python
# นับ Orders ทุก 1 นาที (แต่ละ window ไม่ซ้ำกัน)
.groupBy(
    F.window('event_time', '1 minute'),  # window size
    'product'
).count()
```

```
Timeline: ─────────────────────────────────────────────▶
Events:   e1  e2  e3     e4  e5  e6       e7  e8
          ├──── Window 1 ────┤├──── Window 2 ────┤
          [00:00       01:00] [01:00       02:00]
```

## 4.2 Sliding Window — ซ้อนทับได้

```python
# นับ Orders ทุก 30 วินาที โดยเลื่อน window ทุก 10 วินาที
.groupBy(
    F.window('event_time', '30 seconds', '10 seconds'),  # size, slide
    'category'
).sum('revenue')
```

## 4.3 Watermark — รับมือ Late Data

```python
# ยอมรับข้อมูลที่มาช้าไม่เกิน 10 นาที
stream.withWatermark('event_time', '10 minutes') \
      .groupBy(F.window('event_time', '5 minutes'), 'region') \
      .count()

# สถานการณ์:
# Event เกิดตอน 10:00 แต่ Spark รับได้จนถึง 10:10
# Event ที่มาหลัง 10:10 → ทิ้ง (ไม่นำมาคำนวณ)
```

> 💡 **ทำไม Watermark ถึงสำคัญ?** IoT Sensor หรือ Mobile App อาจส่งข้อมูลช้า (Network Delay) — Watermark กำหนด "รอนานสุดแค่ไหน" เพื่อประหยัด Memory


# 5. 🔬 Lab Setup — ติดตั้งและเตรียมสภาพแวดล้อม (Colab Ready)

> ✅ Cell นี้ทำงานบน **Google Colab ได้โดยตรง** ไม่ต้องติดตั้ง Kafka จริง
> เราจำลอง Stream ด้วย **Socket Server** (แนวทางจาก [Softlandia-Ltd/stateful-streaming-examples](https://github.com/Softlandia-Ltd/stateful-streaming-examples))


In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 0: ติดตั้ง dependencies (รันครั้งเดียว)
# ═══════════════════════════════════════════════════════════
import subprocess, sys

# ติดตั้ง PySpark
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark'], check=True)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json, time, threading, socket, random, os

# สร้าง SparkSession
spark = SparkSession.builder \
    .appName('BigData-Week7-Streaming') \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.streaming.stopGracefullyOnShutdown', 'true') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'✅ Spark {spark.version} พร้อมใช้งาน')
print(f'🖥️  Spark UI: http://localhost:4040 (Local) หรือใช้ Colab proxy')

# ── Helper: ตรวจสอบ Port ว่าง ─────────────────────────────────
def is_port_free(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) != 0

# ── Helper: Socket Server (จำลอง Kafka Producer) ──────────────
class StreamServer:
    '''จำลอง Kafka Producer ผ่าน TCP Socket'''
    def __init__(self, port, gen_fn, n=100, delay=0.4):
        self.port, self.gen_fn, self.n, self.delay = port, gen_fn, n, delay
        self._thread = None

    def _serve(self):
        srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(('localhost', self.port))
        srv.listen(1)
        print(f'  📡 Server listening on port {self.port}...')
        srv.settimeout(30)
        try:
            conn, addr = srv.accept()
            print(f'  ✅ Consumer connected from {addr}')
            for i in range(self.n):
                msg = self.gen_fn() + '\n'
                try:
                    conn.sendall(msg.encode('utf-8'))
                except BrokenPipeError:
                    break
                time.sleep(self.delay)
            conn.close()
        except socket.timeout:
            print(f'  ⚠️ Timeout waiting for consumer on port {self.port}')
        finally:
            srv.close()
            print(f'  ✅ Server on port {self.port} closed')

    def start(self):
        self._thread = threading.Thread(target=self._serve, daemon=True)
        self._thread.start()
        time.sleep(0.5)  # ให้เวลา server เปิด
        return self

print('✅ Helper utilities พร้อมใช้งาน')


# 6. 🏭 Data Generators — จำลอง Kafka Producers

สร้าง Generator 3 แบบสำหรับ Lab ต่างๆ:
1. **E-commerce Orders** — ยอดขายสินค้า
2. **IoT Sensor** — อุณหภูมิเซ็นเซอร์
3. **Financial Transaction** — ธุรกรรมธนาคาร


In [ ]:
# ═══════════════════════════════════════════════════════════
# GENERATORS — จำลองข้อมูลจาก Producer ต่างๆ
# ═══════════════════════════════════════════════════════════

PRODUCTS = ['iPhone 15', 'MacBook Pro', 'AirPods Pro', 'iPad Air', 'Apple Watch']
PRICES   = {'iPhone 15': 35900, 'MacBook Pro': 62900, 'AirPods Pro': 7900,
             'iPad Air': 24900, 'Apple Watch': 14900}
CATS     = {'iPhone 15': 'Phone', 'MacBook Pro': 'Laptop', 'AirPods Pro': 'Audio',
             'iPad Air': 'Tablet', 'Apple Watch': 'Wearable'}
REGIONS  = ['Bangkok', 'Chiang Mai', 'Phuket', 'Pattaya', 'Khon Kaen']
STATUSES = ['SUCCESS', 'SUCCESS', 'SUCCESS', 'FAILED', 'PENDING']  # 60% success

def gen_order():
    '''จำลอง E-commerce Order Event (แทน Kafka Topic: orders)'''
    p = random.choice(PRODUCTS)
    qty = random.randint(1, 3)
    return json.dumps({
        'order_id':  f'ORD-{random.randint(10000, 99999)}',
        'product':   p,
        'category':  CATS[p],
        'quantity':  qty,
        'price':     PRICES[p],
        'revenue':   PRICES[p] * qty,
        'region':    random.choice(REGIONS),
        'status':    random.choice(STATUSES),
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    })

def gen_sensor():
    '''จำลอง IoT Sensor Reading (แทน Kafka Topic: sensor-data)'''
    sensor_id = f'SENSOR-{random.randint(1, 10):02d}'
    # บางครั้งค่าผิดปกติ (anomaly)
    temp = random.uniform(60, 90) if random.random() < 0.1 else random.uniform(20, 35)
    return json.dumps({
        'sensor_id':   sensor_id,
        'location':    random.choice(['Factory-A', 'Factory-B', 'Warehouse']),
        'temperature': round(temp, 2),
        'humidity':    round(random.uniform(40, 80), 2),
        'timestamp':   time.strftime('%Y-%m-%d %H:%M:%S'),
    })

def gen_transaction():
    '''จำลอง Bank Transaction (แทน Kafka Topic: transactions)'''
    is_fraud = random.random() < 0.15  # 15% fraud
    return json.dumps({
        'txn_id':    f'TXN-{random.randint(100000, 999999)}',
        'user_id':   f'U{random.randint(1, 50):03d}',
        'amount':    round(random.uniform(80000, 500000), 2) if is_fraud
                     else round(random.uniform(100, 50000), 2),
        'merchant':  random.choice(['Amazon', 'Lazada', 'Shopee', 'Unknown']),
        'region':    random.choice(REGIONS),
        'is_fraud':  is_fraud,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    })

# ── ทดสอบ Generators ─────────────────────────────────────────
print('📦 ตัวอย่าง Order Event:', gen_order()[:120])
print('🌡️  ตัวอย่าง Sensor Event:', gen_sensor()[:120])
print('💳 ตัวอย่าง Transaction Event:', gen_transaction()[:120])


# 7. 🛒 Lab 1: E-commerce Order Stream — Running Count & Revenue

**สถานการณ์:** ระบบ E-commerce ส่ง Order events เข้า Kafka Topic `orders`
เราต้องวิเคราะห์ **ยอดขายแบบ Real-time** แยกตาม Category

**สิ่งที่จะได้เรียน:**
- `readStream` + `from_json` แปลง JSON → DataFrame
- `groupBy + agg` วิเคราะห์บน Stream
- `outputMode('complete')` แสดงผลทั้งหมดทุก trigger
- `foreachBatch` สำหรับ custom logic


In [ ]:
# ═══════════════════════════════════════════════════════════
# LAB 1: E-Commerce Order Stream
# ═══════════════════════════════════════════════════════════
PORT_ORDERS = 9999

ORDER_SCHEMA = StructType([
    StructField('order_id',  StringType(),  True),
    StructField('product',   StringType(),  True),
    StructField('category',  StringType(),  True),
    StructField('quantity',  IntegerType(), True),
    StructField('price',     DoubleType(),  True),
    StructField('revenue',   DoubleType(),  True),
    StructField('region',    StringType(),  True),
    StructField('status',    StringType(),  True),
    StructField('timestamp', StringType(),  True),
])

# ── เริ่ม Socket Server (จำลอง Kafka Producer) ────────────────
server1 = StreamServer(PORT_ORDERS, gen_order, n=80, delay=0.4).start()

# ── readStream (จำลอง Kafka Consumer) ────────────────────────
raw_stream = spark.readStream \
    .format('socket') \
    .option('host', 'localhost') \
    .option('port', PORT_ORDERS) \
    .load()

# ── Parse JSON + เพิ่ม event_time ────────────────────────────
orders = raw_stream \
    .select(F.from_json(F.col('value'), ORDER_SCHEMA).alias('d')) \
    .select('d.*') \
    .withColumn('event_time', F.to_timestamp('timestamp', 'yyyy-MM-dd HH:mm:ss')) \
    .filter(F.col('status') == 'SUCCESS')  # กรองเฉพาะ SUCCESS (Week5 concept)

print('✅ Stream schema:')
orders.printSchema()


In [ ]:
# ── Query A: Real-time Revenue by Category (Complete Mode) ───
revenue_by_cat = orders \
    .groupBy('category') \
    .agg(
        F.count('order_id').alias('total_orders'),
        F.sum('revenue').alias('total_revenue'),
        F.avg('revenue').alias('avg_order_value')
    ) \
    .orderBy(F.desc('total_revenue'))

q_revenue = revenue_by_cat.writeStream \
    .outputMode('complete') \
    .format('console') \
    .option('truncate', False) \
    .option('numRows', 10) \
    .trigger(processingTime='5 seconds') \
    .start()

print('🔴 Query A: Revenue by Category — กำลังทำงาน...')
print('   (รอประมาณ 30 วินาทีเพื่อดูผลลัพธ์หลาย batches)')
q_revenue.awaitTermination(35)
q_revenue.stop()
print('✅ Query A หยุดทำงาน')


In [ ]:
# ── Query B: foreachBatch — Custom Processing per Micro-batch ─
# Reference: https://medium.com/@canadiandataguy/
#            simplifying-real-time-data-processing-with-
#            spark-streamings-foreachbatch-with-working-code

server1b = StreamServer(PORT_ORDERS, gen_order, n=60, delay=0.4).start()

raw2 = spark.readStream.format('socket') \
    .option('host', 'localhost').option('port', PORT_ORDERS).load()
orders2 = raw2 \
    .select(F.from_json(F.col('value'), ORDER_SCHEMA).alias('d')).select('d.*') \
    .withColumn('event_time', F.to_timestamp('timestamp', 'yyyy-MM-dd HH:mm:ss'))

batch_counter = [0]  # mutable counter

def process_batch(batch_df, batch_id):
    '''foreachBatch: เรียกทุก micro-batch — ใช้ทำ Custom Logic'''
    count = batch_df.count()
    if count == 0:
        return
    batch_counter[0] += 1
    revenue = batch_df.agg(F.sum('revenue')).collect()[0][0] or 0
    top_product = batch_df.groupBy('product').count() \
                          .orderBy(F.desc('count')).first()
    print(f'\n📦 Batch #{batch_id} | Records: {count} | '
          f'Revenue: ฿{revenue:,.0f} | '
          f'Top Product: {top_product["product"] if top_product else "N/A"}')
    # บันทึก Batch ลง Parquet (Week 2 concept)
    if count > 0:
        out = f'/tmp/stream_batch_{batch_id}'
        batch_df.write.mode('overwrite').parquet(out)

q_foreach = orders2.writeStream \
    .foreachBatch(process_batch) \
    .option('checkpointLocation', '/tmp/ckpt_foreach') \
    .trigger(processingTime='5 seconds') \
    .start()

print('🔴 Query B: foreachBatch — กำลังทำงาน...')
q_foreach.awaitTermination(30)
q_foreach.stop()
print(f'\n✅ ประมวลผลครบ {batch_counter[0]} batches')


# 8. 🕐 Lab 2: Window Aggregation + Sink to Parquet

**สถานการณ์:** วิเคราะห์ยอดขายรายช่วงเวลา (Tumbling Window) และบันทึกลง Parquet

**สิ่งที่จะได้เรียน:**
- `withWatermark` + `F.window` สำหรับ Time-based Aggregation
- Sink ลง **Parquet** (เชื่อมกับ Week 2)
- อ่าน Parquet กลับเพื่อวิเคราะห์แบบ Batch (เชื่อมกับ Week 4)


In [ ]:
# ═══════════════════════════════════════════════════════════
# LAB 2: Window Aggregation + Parquet Sink
# ═══════════════════════════════════════════════════════════
PORT_WIN = 9998

ORDER_SCHEMA = StructType([
    StructField('order_id',  StringType(),  True),
    StructField('product',   StringType(),  True),
    StructField('category',  StringType(),  True),
    StructField('quantity',  IntegerType(), True),
    StructField('price',     DoubleType(),  True),
    StructField('revenue',   DoubleType(),  True),
    StructField('region',    StringType(),  True),
    StructField('status',    StringType(),  True),
    StructField('timestamp', StringType(),  True),
])

server2 = StreamServer(PORT_WIN, gen_order, n=120, delay=0.3).start()

stream_w = spark.readStream.format('socket') \
    .option('host', 'localhost').option('port', PORT_WIN).load() \
    .select(F.from_json(F.col('value'), ORDER_SCHEMA).alias('d')).select('d.*') \
    .withColumn('event_time', F.to_timestamp('timestamp', 'yyyy-MM-dd HH:mm:ss'))

# ── Tumbling Window: ยอดขายทุก 30 วินาที ──────────────────────
windowed = stream_w \
    .withWatermark('event_time', '1 minute') \
    .groupBy(
        F.window('event_time', '30 seconds'),  # Tumbling Window
        'category'
    ) \
    .agg(
        F.count('order_id').alias('order_count'),
        F.sum('revenue').alias('window_revenue'),
        F.avg('price').alias('avg_price')
    )

# ── Sink 1: Console (debug) ────────────────────────────────────
q_win_console = windowed.writeStream \
    .outputMode('append') \
    .format('console') \
    .option('truncate', False) \
    .trigger(processingTime='10 seconds') \
    .start()

# ── Raw Sink: บันทึก raw orders ลง Parquet ──────────────────────
stream_w2 = spark.readStream.format('socket') \
    .option('host', 'localhost').option('port', PORT_WIN).load() \
    .select(F.from_json(F.col('value'), ORDER_SCHEMA).alias('d')).select('d.*') \
    .withColumn('event_time', F.to_timestamp('timestamp', 'yyyy-MM-dd HH:mm:ss'))

PARQUET_OUT = '/tmp/streaming_orders_parquet'
q_parquet = stream_w.writeStream \
    .outputMode('append') \
    .format('parquet') \
    .option('path', PARQUET_OUT) \
    .option('checkpointLocation', '/tmp/ckpt_parquet') \
    .trigger(processingTime='10 seconds') \
    .start()

print('🔴 Lab 2: Window + Parquet กำลังทำงาน... (45 วินาที)')
q_win_console.awaitTermination(50)
q_parquet.awaitTermination(50)
q_win_console.stop()
q_parquet.stop()
print('✅ Lab 2 เสร็จสิ้น')


In [ ]:
# ── อ่าน Parquet กลับ — Batch Analysis (เหมือน Week 4-5) ──────
import os

if os.path.exists(PARQUET_OUT) and any(
    f.endswith('.parquet') for root, dirs, files in os.walk(PARQUET_OUT) for f in files
):
    df_result = spark.read.parquet(PARQUET_OUT)
    total = df_result.count()
    print(f'📊 Orders ที่บันทึกลง Parquet: {total:,} rows')
    print()

    print('🏆 Top Products by Revenue:')
    df_result.groupBy('product') \
        .agg(
            F.count('order_id').alias('orders'),
            F.sum('revenue').alias('total_revenue')
        ) \
        .orderBy(F.desc('total_revenue')) \
        .show(10)

    print('\n🗺️  Revenue by Region:')
    df_result.groupBy('region') \
        .agg(F.sum('revenue').alias('revenue')) \
        .orderBy(F.desc('revenue')) \
        .show()
else:
    print('⚠️ ยังไม่มีไฟล์ Parquet — รัน Lab 2 ก่อน')


# 9. 🌡️ Lab 3: IoT Sensor — Anomaly Detection

**สถานการณ์:** โรงงานมีเซ็นเซอร์อุณหภูมิ 10 ตัว ต้องตรวจจับค่าผิดปกติ
(อุณหภูมิ > 55°C = อันตราย) แบบ Real-time

**Reference:** [Stateful Streaming Examples (GitHub)](https://github.com/Softlandia-Ltd/stateful-streaming-examples)


In [ ]:
# ═══════════════════════════════════════════════════════════
# LAB 3: IoT Sensor Anomaly Detection Stream
# ═══════════════════════════════════════════════════════════
PORT_IOT = 9997

SENSOR_SCHEMA = StructType([
    StructField('sensor_id',   StringType(), True),
    StructField('location',    StringType(), True),
    StructField('temperature', DoubleType(), True),
    StructField('humidity',    DoubleType(), True),
    StructField('timestamp',   StringType(), True),
])

server3 = StreamServer(PORT_IOT, gen_sensor, n=100, delay=0.3).start()

sensor_stream = spark.readStream.format('socket') \
    .option('host', 'localhost').option('port', PORT_IOT).load() \
    .select(F.from_json(F.col('value'), SENSOR_SCHEMA).alias('d')).select('d.*') \
    .withColumn('event_time', F.to_timestamp('timestamp', 'yyyy-MM-dd HH:mm:ss'))

# ── Rule 1: Alert เมื่ออุณหภูมิ > 55°C ────────────────────────
anomalies = sensor_stream \
    .filter(F.col('temperature') > 55.0) \
    .select(
        'sensor_id', 'location', 'temperature', 'humidity', 'timestamp',
        F.lit('🚨 ALERT: HIGH TEMP').alias('alert_type')
    )

# ── Rule 2: Sliding Window — ค่าเฉลี่ยต่อ Location ทุก 30 วิ ─
avg_temp = sensor_stream \
    .withWatermark('event_time', '2 minutes') \
    .groupBy(
        F.window('event_time', '30 seconds', '10 seconds'),  # Sliding Window
        'location'
    ) \
    .agg(
        F.avg('temperature').alias('avg_temp'),
        F.max('temperature').alias('max_temp'),
        F.count('sensor_id').alias('readings')
    ) \
    .filter(F.col('avg_temp') > 30)  # แสดงเฉพาะที่น่าสนใจ

# ── Start both queries ────────────────────────────────────────
q_alert = anomalies.writeStream \
    .outputMode('append') \
    .format('console') \
    .option('truncate', False) \
    .trigger(processingTime='5 seconds') \
    .start()

q_avg = avg_temp.writeStream \
    .outputMode('append') \
    .format('console') \
    .option('truncate', False) \
    .trigger(processingTime='10 seconds') \
    .start()

print('🔴 Lab 3: IoT Anomaly Detection กำลังทำงาน... (40 วินาที)')
print('   🌡️  จะแสดง ALERT เมื่อพบอุณหภูมิ > 55°C')
q_alert.awaitTermination(45)
q_avg.awaitTermination(45)
q_alert.stop()
q_avg.stop()
print('✅ Lab 3 เสร็จสิ้น')


# 🔥 Lab 4 (Challenge): Real-time Fraud Detection

**สถานการณ์:** ธนาคารต้องการตรวจจับ Fraud แบบ Real-time ด้วย 2 กฎ:
1. **High Amount:** ยอดเงิน > 50,000 บาท → แจ้งเตือนทันที
2. **Velocity Check:** ธุรกรรมจาก Region เดียวกัน > 5 ครั้งใน 30 วินาที → น่าสงสัย

**📝 โจทย์:** เติมโค้ดในส่วนที่ขาดหายไป (TODO)


In [ ]:
# ═══════════════════════════════════════════════════════════
# LAB 4 CHALLENGE: Fraud Detection
# ═══════════════════════════════════════════════════════════
PORT_TXN = 9996

TXN_SCHEMA = StructType([
    StructField('txn_id',    StringType(), True),
    StructField('user_id',   StringType(), True),
    StructField('amount',    DoubleType(), True),
    StructField('merchant',  StringType(), True),
    StructField('region',    StringType(), True),
    StructField('is_fraud',  BooleanType(), True),
    StructField('timestamp', StringType(), True),
])

# TODO 1: เริ่ม StreamServer ด้วย gen_transaction บน PORT_TXN
# server4 = StreamServer(???).start()

# TODO 2: readStream จาก socket PORT_TXN
# txn_stream = spark.readStream...

# TODO 3: Parse JSON + เพิ่ม event_time
# txns = txn_stream.select(...)

# TODO 4: กรอง High Amount Fraud (amount > 50000)
# high_amount_fraud = txns.filter(...)

# TODO 5: Velocity Check — count per region per 30 seconds
# velocity_fraud = txns \
#     .withWatermark('event_time', '1 minute') \
#     .groupBy(F.window('event_time', '30 seconds'), 'region') \
#     .agg(F.count('txn_id').alias('txn_count')) \
#     .filter(F.col('txn_count') > 5)

# TODO 6: writeStream สำหรับทั้ง 2 queries แล้ว awaitTermination(50)

print('💻 เติมโค้ด TODO 1-6 แล้วรัน cell นี้')


In [ ]:
# ── เฉลย Lab 4 ───────────────────────────────────────────────
PORT_TXN = 9996

TXN_SCHEMA = StructType([
    StructField('txn_id',    StringType(), True),
    StructField('user_id',   StringType(), True),
    StructField('amount',    DoubleType(), True),
    StructField('merchant',  StringType(), True),
    StructField('region',    StringType(), True),
    StructField('is_fraud',  BooleanType(), True),
    StructField('timestamp', StringType(), True),
])

# TODO 1: Server
server4 = StreamServer(PORT_TXN, gen_transaction, n=120, delay=0.35).start()

# TODO 2+3: readStream + parse
txns = spark.readStream.format('socket') \
    .option('host', 'localhost').option('port', PORT_TXN).load() \
    .select(F.from_json(F.col('value'), TXN_SCHEMA).alias('d')).select('d.*') \
    .withColumn('event_time', F.to_timestamp('timestamp', 'yyyy-MM-dd HH:mm:ss'))

# TODO 4: High Amount
high_amount_fraud = txns \
    .filter(F.col('amount') > 50000) \
    .select('txn_id','user_id','amount','merchant','region','timestamp',
            F.lit('🚨 HIGH AMOUNT FRAUD').alias('alert'))

# TODO 5: Velocity Check
velocity_fraud = txns \
    .withWatermark('event_time', '1 minute') \
    .groupBy(F.window('event_time', '30 seconds'), 'region') \
    .agg(F.count('txn_id').alias('txn_count'),
         F.sum('amount').alias('total_amount')) \
    .filter(F.col('txn_count') > 5) \
    .select(
        F.col('window.start').alias('window_start'),
        F.col('window.end').alias('window_end'),
        'region', 'txn_count', 'total_amount',
        F.lit('⚡ VELOCITY FRAUD').alias('alert')
    )

# TODO 6: writeStream
q_ha = high_amount_fraud.writeStream.outputMode('append') \
    .format('console').option('truncate', False) \
    .trigger(processingTime='5 seconds').start()

q_vel = velocity_fraud.writeStream.outputMode('append') \
    .format('console').option('truncate', False) \
    .trigger(processingTime='10 seconds').start()

print('🚨 Fraud Detection กำลังทำงาน... (50 วินาที)')
print('   Rule 1: High Amount Alert > ฿50,000')
print('   Rule 2: Velocity Check > 5 txns/30s per region')
q_ha.awaitTermination(55)
q_vel.awaitTermination(55)
q_ha.stop(); q_vel.stop()
print('✅ Fraud Detection หยุดทำงาน')


# 10. 🚀 Real Kafka Lab — ประสบการณ์ Kafka จริง

> ส่วนนี้ให้นักศึกษา **ใช้ Kafka จริง** ไม่ใช่การจำลอง!  

เลือก **Track ที่เหมาะกับคุณ**:

| Track | สภาพแวดล้อม | ข้อดี | เวลาติดตั้ง |
|-------|------------|-------|------------|
| **Track A 🟢** | Google Colab | ไม่ต้องติดตั้งอะไรเพิ่ม | ~60 วินาที |
| **Track B 🔵** | Local Docker | ใกล้เคียง Production ที่สุด | ~3 นาที |

---


## 🟢 Track A — Kafka บน Google Colab (KRaft Mode)

> **KRaft** = Kafka Raft — Kafka 3.x สามารถรันได้โดย**ไม่ต้องมี Zookeeper**  
> เหมาะสำหรับทดลองบน Colab เพราะเบา เร็ว และติดตั้งง่าย

### ขั้นตอนการทำงาน

```
 [Colab Runtime]
      │
      ├── Step 1: Install kafka-python + download Kafka 3.9.0 binary
      ├── Step 2: Format storage (KRaft) + start broker in background
      ├── Step 3: Create topic  "orders"
      ├── Step 4: Python Producer → sends 30 messages
      ├── Step 5: Python Consumer → reads & prints messages
      └── Step 6: Spark Structured Streaming reads from Kafka topic
```


In [ ]:
# ═══════════════════════════════════════════════════════════
# TRACK A — Step 1: ติดตั้ง kafka-python และ download Kafka binary
# ═══════════════════════════════════════════════════════════
import subprocess, os, sys

# ── 1a) ติดตั้ง Python client (kafka-python-ng รองรับ Python 3.12) ──
print('📦 Installing kafka-python-ng...')
# kafka-python-ng: maintained fork fully compatible with Python 3.12
# Drop-in replacement — same API: from kafka import KafkaProducer / KafkaConsumer
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kafka-python-ng'], check=True)
print('✅ kafka-python-ng installed (Python 3.12 compatible)')

# ── 1b) Download Kafka 3.7.2 binary ───────────────────────
# ใช้ archive.apache.org ซึ่งเก็บทุก version อย่างน่าเชื่อถือ (HTTP 200 verified)
KAFKA_VERSION = '3.7.2'
SCALA_VERSION = '2.13'
KAFKA_DIR = f'/tmp/kafka_{SCALA_VERSION}-{KAFKA_VERSION}'
KAFKA_TGZ = f'/tmp/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz'

# Mirror list — ลองตามลำดับ
MIRRORS = [
    f'https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz',
    f'https://downloads.apache.org/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz',
]

if not os.path.exists(KAFKA_DIR):
    print(f'⬇️  Downloading Kafka {KAFKA_VERSION} (~115 MB)...')
    downloaded = False
    for url in MIRRORS:
        print(f'   Trying: {url}')
        result = subprocess.run(['wget', '-q', '--tries=3', '-O', KAFKA_TGZ, url])
        if result.returncode == 0 and os.path.exists(KAFKA_TGZ) and os.path.getsize(KAFKA_TGZ) > 1_000_000:
            downloaded = True
            print(f'   ✅ Downloaded ({os.path.getsize(KAFKA_TGZ)//1_000_000} MB)')
            break
        else:
            print('   ⚠️  Failed, trying next mirror...')
            if os.path.exists(KAFKA_TGZ):
                os.remove(KAFKA_TGZ)
    if not downloaded:
        raise RuntimeError('❌ All mirrors failed. Check internet connection and retry.')
    print('📦 Extracting...')
    subprocess.run(['tar', '-xzf', KAFKA_TGZ, '-C', '/tmp'], check=True)
    print('✅ Kafka downloaded and extracted')
else:
    print('✅ Kafka already downloaded')

print(f'📂 Kafka directory: {KAFKA_DIR}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# TRACK A — Step 2: ตั้งค่า KRaft และ Start Kafka Broker
# ═══════════════════════════════════════════════════════════
import subprocess, os, time, socket

KAFKA_DIR   = "/tmp/kafka_2.13-3.7.2"
LOG_DIR     = "/tmp/kafka-logs"
CONF_FILE   = f"{KAFKA_DIR}/config/kraft/reconfig-server.properties"
BROKER_PORT = 9092
CTRL_PORT   = 9093

# ── Helper: check if a port is listening ──────────────────
def port_open(port):
    try:
        with socket.create_connection(("localhost", port), timeout=1):
            return True
    except OSError:
        return False

# ── Step 2a) Kill any lingering Kafka JVM process ─────────
# Re-running this cell without killing causes port 9093 conflict
kill = subprocess.run(
    ["pkill", "-9", "-f", "kafka-server-start"],
    capture_output=True
)
if kill.returncode == 0:
    print("🔴 Killed previous Kafka process — waiting for ports to free...")
    for _ in range(20):
        time.sleep(1)
        if not port_open(BROKER_PORT) and not port_open(CTRL_PORT):
            print("✅ Ports 9092 & 9093 are free")
            break
    else:
        print("⚠️  Ports still busy after 20s — proceeding anyway")
else:
    print("ℹ️  No previous Kafka process found")

# ── Step 2b) Write broker config (always recreate) ────────
os.makedirs(f"{KAFKA_DIR}/config/kraft", exist_ok=True)
with open(CONF_FILE, "w") as cf:
    cf.write(
        "process.roles=broker,controller\n"
        "node.id=1\n"
        "controller.quorum.voters=1@localhost:9093\n"
        "listeners=PLAINTEXT://localhost:9092,CONTROLLER://localhost:9093\n"
        "inter.broker.listener.name=PLAINTEXT\n"
        "advertised.listeners=PLAINTEXT://localhost:9092\n"
        "controller.listener.names=CONTROLLER\n"
        "listener.security.protocol.map=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT\n"
        f"log.dirs={LOG_DIR}\n"
        "num.partitions=3\n"
        "offsets.topic.replication.factor=1\n"
        "transaction.state.log.replication.factor=1\n"
        "transaction.state.log.min.isr=1\n"
        "auto.create.topics.enable=true\n"
    )
print("✅ Config written")

# ── Step 2c) Format storage (only if not already done) ────
# Use meta.properties as reliable "already formatted" indicator
meta_file = os.path.join(LOG_DIR, "meta.properties")
if not os.path.exists(meta_file):
    os.makedirs(LOG_DIR, exist_ok=True)
    cluster_id = subprocess.check_output(
        [f"{KAFKA_DIR}/bin/kafka-storage.sh", "random-uuid"]
    ).decode().strip()
    print(f"🔑 Cluster UUID: {cluster_id}")
    subprocess.run([
        f"{KAFKA_DIR}/bin/kafka-storage.sh", "format",
        "-t", cluster_id, "-c", CONF_FILE
    ], check=True, capture_output=True)
    print("✅ Storage formatted")
else:
    print("✅ Storage already formatted — skipping format step")

# ── Step 2d) Start broker in background ───────────────────
log_f = open("/tmp/kafka-broker.log", "w")
proc  = subprocess.Popen(
    [f"{KAFKA_DIR}/bin/kafka-server-start.sh", CONF_FILE],
    stdout=log_f, stderr=log_f
)
print(f"🚀 Kafka broker starting (PID={proc.pid})...")

# ── Step 2e) Poll both ports — broker(9092) + ctrl(9093) ──
for i in range(45):
    time.sleep(1)
    b_up = port_open(BROKER_PORT)
    c_up = port_open(CTRL_PORT)
    if b_up and c_up:
        print(f"✅ Kafka is UP  broker:{BROKER_PORT} ✔  controller:{CTRL_PORT} ✔  ({i+1}s)")
        break
    if i > 0 and i % 10 == 0:
        print(f"   waiting... {i}s  broker={b_up}  ctrl={c_up}")
else:
    with open("/tmp/kafka-broker.log") as lf:
        lines = lf.read().splitlines()
    print("⚠️  Broker did not start within 45s. Last log lines:")
    print("\n".join(lines[-20:]))


In [ ]:
# ═══════════════════════════════════════════════════════════
# TRACK A — Step 3: สร้าง Kafka Topic 'orders'
# ═══════════════════════════════════════════════════════════
import subprocess, time

KAFKA_DIR   = '/tmp/kafka_2.13-3.7.2'
BOOTSTRAP   = 'localhost:9092'
TOPIC_NAME  = 'orders'

time.sleep(3)  # buffer

result = subprocess.run([
    f'{KAFKA_DIR}/bin/kafka-topics.sh',
    '--bootstrap-server', BOOTSTRAP,
    '--create',
    '--topic', TOPIC_NAME,
    '--partitions', '3',
    '--replication-factor', '1',
    '--if-not-exists'
], capture_output=True, text=True)

print(result.stdout or result.stderr)

# List topics
list_result = subprocess.run([
    f'{KAFKA_DIR}/bin/kafka-topics.sh',
    '--bootstrap-server', BOOTSTRAP,
    '--list'
], capture_output=True, text=True)

print('📋 Topics on broker:')
print(list_result.stdout)


In [ ]:
# ═══════════════════════════════════════════════════════════
# TRACK A — Step 4: Python Producer — ส่ง 30 E-Commerce Orders
# ═══════════════════════════════════════════════════════════
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

BOOTSTRAP  = 'localhost:9092'
TOPIC_NAME = 'orders'

CATEGORIES  = ['Electronics', 'Clothing', 'Books', 'Sports', 'Food']
PRODUCTS    = {
    'Electronics': ['Laptop', 'Phone', 'Tablet', 'Headphones'],
    'Clothing':    ['T-Shirt', 'Jeans', 'Jacket', 'Sneakers'],
    'Books':       ['Python Book', 'ML Book', 'History', 'Finance'],
    'Sports':      ['Tennis Racket', 'Football', 'Yoga Mat', 'Weights'],
    'Food':        ['Coffee', 'Chocolate', 'Tea', 'Protein Bar'],
}

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8')
)

print(f'📤 Sending 30 messages to topic "{TOPIC_NAME}"...')
sent = []
for i in range(30):
    category = random.choice(CATEGORIES)
    product  = random.choice(PRODUCTS[category])
    msg = {
        'order_id':  f'ORD-{1000+i}',
        'product':   product,
        'category':  category,
        'price':     round(random.uniform(10, 2000), 2),
        'quantity':  random.randint(1, 5),
        'timestamp': datetime.now().isoformat(),
    }
    producer.send(TOPIC_NAME, key=msg['order_id'], value=msg)
    sent.append(msg)
    time.sleep(0.1)

producer.flush()
producer.close()

print(f'✅ {len(sent)} messages sent!')
print('\n📊 Sample messages:')
for m in sent[:5]:
    print(f"  [{m['order_id']}] {m['product']:12s} ({m['category']:12s}) ฿{m['price']:8.2f} x{m['quantity']}")


In [ ]:
# ═══════════════════════════════════════════════════════════
# TRACK A — Step 5: Python Consumer — อ่าน messages จาก Kafka
# ═══════════════════════════════════════════════════════════
from kafka import KafkaConsumer
import json

BOOTSTRAP  = 'localhost:9092'
TOPIC_NAME = 'orders'

consumer = KafkaConsumer(
    TOPIC_NAME,
    bootstrap_servers=BOOTSTRAP,
    auto_offset_reset='earliest',       # อ่านตั้งแต่ต้น
    consumer_timeout_ms=5000,           # หยุดรอ 5 วินาทีถ้าไม่มี message ใหม่
    group_id='colab-consumer-group-1',  # Consumer Group
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
    key_deserializer=lambda k: k.decode('utf-8') if k else None,
)

print(f'📥 Consuming from topic "{TOPIC_NAME}" (Consumer Group: colab-consumer-group-1)...')
print('-' * 70)

records = []
for msg in consumer:
    records.append(msg.value)
    print(f"  Partition={msg.partition} Offset={msg.offset:3d} "
          f"Key={msg.key:10s} | {msg.value['product']:12s} ฿{msg.value['price']:.2f}")

consumer.close()
print('-' * 70)
print(f'\n✅ Total records consumed: {len(records)}')
total_rev = sum(r['price'] * r['quantity'] for r in records)
print(f'💰 Total Revenue: ฿{total_rev:,.2f}')

# Category Summary
from collections import Counter
cats = Counter(r['category'] for r in records)
print('\n📊 Orders by Category:')
for cat, count in cats.most_common():
    print(f'  {cat:12s}: {count} orders')


In [ ]:
# ═══════════════════════════════════════════════════════════
# TRACK A — Step 6: Spark Structured Streaming reads from Real Kafka
# ═══════════════════════════════════════════════════════════
import subprocess, sys

# ── ติดตั้ง PySpark + Kafka connector ────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyspark==3.5.1'], check=True)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

# ── สร้าง SparkSession พร้อม Kafka package ────────────────────
spark = (SparkSession.builder
    .appName('RealKafkaColabLab')
    .config('spark.jars.packages',
            'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('✅ SparkSession ready')

# ── Schema สำหรับ parse JSON payload ─────────────────────────
ORDER_SCHEMA = StructType([
    StructField('order_id',  StringType()),
    StructField('product',   StringType()),
    StructField('category',  StringType()),
    StructField('price',     DoubleType()),
    StructField('quantity',  IntegerType()),
    StructField('timestamp', StringType()),
])

# ── อ่าน Stream จาก Kafka ─────────────────────────────────────
kafka_df = (spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', 'localhost:9092')
    .option('subscribe', 'orders')
    .option('startingOffsets', 'earliest')
    .load())

# ── Parse JSON value ──────────────────────────────────────────
orders = (kafka_df
    .select(F.col('key').cast('string').alias('key'),
            F.from_json(F.col('value').cast('string'), ORDER_SCHEMA).alias('data'),
            'partition', 'offset', 'timestamp')
    .select('key', 'data.*', 'partition', 'offset',
            F.col('timestamp').alias('kafka_ts')))

# ── Aggregation: Revenue by Category ─────────────────────────
revenue_df = (orders
    .withColumn('revenue', F.col('price') * F.col('quantity'))
    .groupBy('category')
    .agg(
        F.count('*').alias('orders'),
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.round(F.avg('price'), 2).alias('avg_price')
    )
    .orderBy(F.desc('total_revenue')))

# ── Start Streaming Query ─────────────────────────────────────
print('🔥 Starting Spark Structured Streaming from Real Kafka...')
query = (revenue_df.writeStream
    .outputMode('complete')
    .format('console')
    .option('truncate', False)
    .option('numRows', 10)
    .trigger(once=True)   # รันครั้งเดียวแล้วหยุด (สำหรับ Colab)
    .start())

query.awaitTermination(60)
print('\n✅ Streaming query completed!')
spark.stop()


---

## 🔵 Track B — Kafka บน Local Machine ด้วย Docker

> รันบน **laptop ของคุณ** ประสบการณ์ใกล้เคียง Production มากที่สุด  
> ต้องการ: **Docker Desktop** เท่านั้น (ไม่ต้องติดตั้ง Kafka เอง)

### Stack ที่จะรัน

| Container | Port | สิ่งที่ทำ |
|-----------|------|----------|
| `kafka`   | 9092 | Kafka broker (KRaft mode) |
| `kafka-ui`| 8080 | Web UI สำหรับดู topics/messages |

### ขั้นตอน

```
Step 1: docker compose up -d         ← เปิด Kafka + Kafka-UI
Step 2: เปิด http://localhost:8080   ← ดู Kafka-UI
Step 3: สร้าง topic ผ่าน CLI หรือ UI
Step 4: รัน Python Producer
Step 5: รัน Python Consumer
Step 6: รัน Spark Streaming job
```


In [ ]:
# ═══════════════════════════════════════════════════════════
# TRACK B — Step 1: สร้าง docker-compose.yml
# รันใน terminal ของ local machine (ไม่ใช่ Colab)
# ═══════════════════════════════════════════════════════════

DOCKER_COMPOSE = '''\
version: "3.8"

services:
  kafka:
    image: apache/kafka:3.9.0
    container_name: kafka
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://localhost:9092
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
      KAFKA_AUTO_CREATE_TOPICS_ENABLE: "true"
      CLUSTER_ID: MkU3OEVBNTcwNTJENDM2Qk
    healthcheck:
      test: ["CMD", "/opt/kafka/bin/kafka-topics.sh", "--bootstrap-server", "localhost:9092", "--list"]
      interval: 10s
      retries: 5

  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    container_name: kafka-ui
    ports:
      - "8080:8080"
    environment:
      KAFKA_CLUSTERS_0_NAME: local
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: kafka:9092
    depends_on:
      kafka:
        condition: service_healthy
'''

# บันทึก docker-compose.yml
with open('docker-compose.yml', 'w') as f:
    f.write(DOCKER_COMPOSE)

print('✅ docker-compose.yml created!')
print(DOCKER_COMPOSE)


### 🖥️ คำสั่ง Terminal — เปิดใช้ Kafka Local

เปิด **Terminal** บน laptop แล้วรันทีละขั้น:

#### Step 1 — เปิด Kafka + UI
```bash
# ไปที่ folder ที่มี docker-compose.yml
cd /path/to/your/project

# เปิด containers (-d คือ background)
docker compose up -d

# ดู status
docker compose ps
```

#### Step 2 — เปิด Kafka-UI ใน Browser
```
http://localhost:8080
```
คุณจะเห็น Dashboard แสดง cluster, topics, consumers

#### Step 3 — สร้าง Topic ผ่าน CLI
```bash
docker exec kafka /opt/kafka/bin/kafka-topics.sh \\
  --bootstrap-server localhost:9092 \\
  --create \\
  --topic orders \\
  --partitions 3 \\
  --replication-factor 1

# ดู topics ทั้งหมด
docker exec kafka /opt/kafka/bin/kafka-topics.sh \\
  --bootstrap-server localhost:9092 \\
  --list
```

#### Step 4 — Console Producer (ทดสอบเบื้องต้น)
```bash
# พิมพ์ message แล้วกด Enter ส่งทีละ message
docker exec -it kafka /opt/kafka/bin/kafka-console-producer.sh \\
  --bootstrap-server localhost:9092 \\
  --topic orders
# พิมพ์: hello world
# กด Ctrl+C เพื่อออก
```

#### Step 5 — Console Consumer (ทดสอบเบื้องต้น)
```bash
docker exec -it kafka /opt/kafka/bin/kafka-console-consumer.sh \\
  --bootstrap-server localhost:9092 \\
  --topic orders \\
  --from-beginning
# กด Ctrl+C เพื่อออก
```

#### Step 6 — ปิด Kafka เมื่อเสร็จ
```bash
docker compose down
```


In [ ]:
# ═══════════════════════════════════════════════════════════
# TRACK B — Step 7: Python Producer + Consumer สำหรับ Local Docker
# รันใน Jupyter Notebook หรือ .py file บน local machine
# ต้องมี: pip install kafka-python
# ═══════════════════════════════════════════════════════════

LOCAL_KAFKA_CODE = '''
# ── Producer ─────────────────────────────────────────────────
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8"),
)

categories = ["Electronics", "Clothing", "Books", "Sports", "Food"]

for i in range(50):
    msg = {
        "order_id":  f"ORD-{2000+i}",
        "category":  random.choice(categories),
        "price":     round(random.uniform(10, 5000), 2),
        "quantity":  random.randint(1, 10),
        "timestamp": datetime.now().isoformat(),
    }
    future = producer.send("orders", key=msg["order_id"], value=msg)
    result = future.get(timeout=10)
    print(f"Sent to Partition={result.partition} Offset={result.offset}: {msg[\"order_id\"]}")
    time.sleep(0.2)

producer.flush()
producer.close()
print("Producer done!")

# ── Consumer (run in a separate cell/file) ────────────────────
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    "orders",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",
    consumer_timeout_ms=10000,
    group_id="local-consumer-group",
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    key_deserializer=lambda k: k.decode("utf-8") if k else None,
)

total = 0
for msg in consumer:
    v = msg.value
    print(f"[P{msg.partition}:O{msg.offset}] {v[\"order_id\"]:10s} {v[\"category\"]:12s} ฿{v[\"price\"]:8.2f}")
    total += 1

print(f"Total consumed: {total} messages")
consumer.close()
'''

print('📋 Python Producer + Consumer code for Local Docker Kafka:')
print('   (Copy and run this on your local machine with Docker Kafka running)')
print()
print(LOCAL_KAFKA_CODE)


In [ ]:
# ═══════════════════════════════════════════════════════════
# TRACK B — Step 8: Spark Structured Streaming + Local Docker Kafka
# รันใน local Jupyter หรือ spark-submit บน terminal
# ═══════════════════════════════════════════════════════════

SPARK_LOCAL_CODE = '''
# spark-submit command:
# spark-submit \\
#   --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1 \\
#   spark_kafka_stream.py

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (SparkSession.builder
    .appName("LocalKafkaStream")
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

ORDER_SCHEMA = StructType([
    StructField("order_id",  StringType()),
    StructField("category",  StringType()),
    StructField("price",     DoubleType()),
    StructField("quantity",  IntegerType()),
    StructField("timestamp", StringType()),
])

# ── อ่าน Stream จาก Kafka ────────────────────────────────────
raw = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "latest")   # ฟัง messages ใหม่เท่านั้น
    .load())

orders = (raw
    .select(F.from_json(F.col("value").cast("string"), ORDER_SCHEMA).alias("d"),
            F.col("timestamp").alias("event_time"))
    .select("d.*", "event_time")
    .withColumn("event_time", F.col("event_time").cast("timestamp")))

# ── Windowed Revenue (Tumbling 1-minute window) ──────────────
windowed = (orders
    .withWatermark("event_time", "30 seconds")
    .groupBy(
        F.window("event_time", "1 minute"),
        "category"
    )
    .agg(
        F.count("*").alias("orders"),
        F.round(F.sum(F.col("price") * F.col("quantity")), 2).alias("revenue")
    ))

# ── เขียนผลลัพธ์ไปที่ Console ────────────────────────────────
query = (windowed.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .start())

query.awaitTermination()  # Ctrl+C เพื่อหยุด
'''

print('📋 Spark Structured Streaming code for Local Docker Kafka:')
print(SPARK_LOCAL_CODE)


# 11. 🔗 Integration: Airflow + Spark Streaming (Week 6 → 7)

> **เชื่อมกับ Week 6:** Airflow เป็น Orchestrator ที่ควบคุม Kafka + Spark Streaming

```
Airflow DAG: streaming_pipeline_daily

  ┌──────────────────┐
  │  check_kafka_    │
  │  health          │  (HttpSensor)
  └────────┬─────────┘
           │
  ┌────────▼─────────┐     ┌───────────────────┐
  │  start_producer  │────▶│  spark_streaming_ │
  │  (BashOperator)  │     │  job              │
  └──────────────────┘     │(SparkSubmitOp.)   │
                           └────────┬──────────┘
                                    │
                           ┌────────▼──────────┐
                           │  check_output_    │
                           │  parquet          │
                           │  (FileSensor)     │
                           └────────┬──────────┘
                                    │
                           ┌────────▼──────────┐
                           │  trigger_eda_     │
                           │  report (Week 8)  │
                           └───────────────────┘
```


In [ ]:
# ═══════════════════════════════════════════════════════════
# Airflow DAG: Orchestrate Kafka + Spark Streaming
# ═══════════════════════════════════════════════════════════
AIRFLOW_DAG = """
from airflow import DAG
from airflow.providers.apache.spark.operators.spark_submit import SparkSubmitOperator
from airflow.operators.bash import BashOperator
from airflow.sensors.filesystem import FileSensor
from airflow.sensors.http_sensor import HttpSensor
from datetime import datetime, timedelta

default_args = {
    "owner": "data-engineer",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "email_on_failure": True,
    "email": ["alert@company.com"]
}

with DAG(
    "streaming_pipeline",
    default_args=default_args,
    schedule_interval="@hourly",
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=["streaming", "kafka", "spark"]
) as dag:

    # 1. ตรวจสอบ Kafka ทำงานปกติ
    check_kafka = HttpSensor(
        task_id="check_kafka_health",
        http_conn_id="kafka_admin",
        endpoint="/v3/clusters",
        timeout=30
    )

    # 2. เปิด Data Producer
    start_producer = BashOperator(
        task_id="start_kafka_producer",
        bash_command="""
            python /opt/producers/ecommerce_producer.py \\
            --topic ecommerce-orders \\
            --duration 3600 &  # รัน 1 ชั่วโมง
            echo $! > /tmp/producer.pid
        """
    )

    # 3. รัน Spark Streaming Job
    spark_streaming = SparkSubmitOperator(
        task_id="spark_streaming_job",
        application="/opt/spark_jobs/streaming_consumer.py",
        packages="org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0",
        conf={
            "spark.streaming.stopGracefullyOnShutdown": "true",
            "spark.sql.shuffle.partitions": "10",
        },
        application_args=["--duration", "3600"],
        execution_timeout=timedelta(hours=2)
    )

    # 4. รอให้ Parquet ออกมา
    check_output = FileSensor(
        task_id="check_parquet_output",
        filepath="/data/streaming/orders/",
        timeout=300,
        poke_interval=30
    )

    # 5. Trigger EDA Report (Week 8)
    trigger_eda = BashOperator(
        task_id="trigger_eda_report",
        bash_command="jupyter nbconvert --to html BigData_Week8_EDA.ipynb --execute"
    )

    check_kafka >> start_producer >> spark_streaming >> check_output >> trigger_eda
"""

print("📝 Airflow DAG สำหรับ Streaming Pipeline (Week 6 Concept):")
print(AIRFLOW_DAG[:500], "...")


# 12. 🏗️ Modern Big Data Architecture — ภาพรวมครบวงจร


<div align="center">
  <img src="https://github.com/witsarutsarai12-Academic/128-356-Big-Data/blob/main/images/week7/modern_data_stack.png?raw=1" width="800" alt="Modern Big Data Architecture: Data Sources → Kafka → Spark (Batch+Streaming) → Airflow → Data Lake → Analytics (Week 1-8)">
  <br><i>Modern Big Data Architecture: Data Sources → Kafka → Spark (Batch+Streaming) → Airflow → Data Lake → Analytics (Week 1-8)</i>
</div>


## สรุปการทำงานร่วมกันของทุกเครื่องมือ

| Layer | ชั้น | เครื่องมือ | สัปดาห์ | บทบาท |
|-------|-----|----------|---------|--------|
| **Ingestion** | รับข้อมูล | Apache Kafka | W7 | รับ Events จาก Producer |
| **Processing** | ประมวลผล | Apache Spark | W4, W7 | Batch + Streaming Analytics |
| **Orchestration** | ควบคุม | Apache Airflow | W6 | Trigger/Monitor ทุก Job |
| **Storage** | จัดเก็บ | Parquet/S3 | W2, W5 | Data Lake |
| **Query** | สืบค้น | DuckDB/SparkSQL | W3 | Business Queries |
| **Analytics** | วิเคราะห์ | EDA/KPI/Charts | W8 | Dashboard, Reports |

### Data Flow ตัวอย่าง: Order ลูกค้า 1 รายการ

```
1. ลูกค้าซื้อสินค้า → Web App ส่ง Event
2. Kafka Producer รับ → ส่งลง Topic 'orders' Partition 2
3. Spark Streaming อ่าน Offset 1234 → วิเคราะห์ window ทุก 30s
4. บันทึกลง Parquet (/datalake/orders/year=2024/month=03/)
5. Airflow ตรวจสอบงาน → Trigger EDA รายวัน
6. DuckDB Query → KPI Dashboard อัปเดต
```


# 13. ⚡ Performance Tips — เร่งความเร็ว Streaming

## จาก GitHub: [polomarcus/Spark-Structured-Streaming-Examples](https://github.com/polomarcus/Spark-Structured-Streaming-Examples)

| เทคนิค | วิธีทำ | ผลลัพธ์ |
|--------|--------|--------|
| **Partition tuning** | `spark.sql.shuffle.partitions = 4-8` (ไม่ใช่ default 200) | เร็วขึ้นมากสำหรับ stream เล็ก |
| **Trigger interval** | `processingTime='10 seconds'` | ลด overhead ต่อ batch |
| **maxOffsetsPerTrigger** | จำกัดจำนวน records ต่อ batch | ป้องกัน memory overflow |
| **Watermark** | ตั้งให้พอดี (ไม่ใหญ่เกิน) | ลด state memory |
| **Checkpoint** | เก็บลง Fast Storage (SSD/S3) | Recovery เร็ว |
| **Coalesce** | `.coalesce(1)` ก่อน write ไฟล์เล็ก | ลดจำนวนไฟล์ |

```python
# ตัวอย่าง: Optimized Spark Session สำหรับ Streaming
spark = SparkSession.builder \
    .appName('OptimizedStreaming') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.streaming.stopGracefullyOnShutdown', 'true') \
    .config('spark.sql.streaming.checkpointLocation', '/tmp/checkpoints') \
    .getOrCreate()

# จำกัด records ต่อ Trigger (Kafka)
kafka_stream = spark.readStream.format('kafka') \
    .option('maxOffsetsPerTrigger', 10000) \
    ...load()
```


# 📋 สรุปบทเรียน Week 7

## สิ่งที่เรียนรู้วันนี้

| หัวข้อ | สาระสำคัญ | Lab |
|--------|----------|-----|
| **Batch vs Streaming** | Lambda/Kappa Architecture | Section 1 |
| **Apache Kafka** | Producer→Topic/Partition→Consumer Group | Section 2 |
| **Spark Structured Streaming** | readStream→Transform→writeStream | Section 3 |
| **Window & Watermark** | Tumbling, Sliding, Late Data | Section 4 |
| **E-commerce Lab** | foreachBatch, Custom Logic | Lab 1 |
| **IoT Lab** | Anomaly Detection, Sliding Window | Lab 3 |
| **Fraud Detection** | Multi-rule Real-time Alert | Lab 4 |
| **Kafka Production** | Exactly-once, Consumer Groups | Section 10 |
| **Airflow Integration** | Orchestrate Streaming Jobs | Section 11 |

## Cheat Sheet — คำสั่งสำคัญ

```python
# READ STREAM
df = spark.readStream.format('socket' | 'kafka' | 'json') \
     .option('host','localhost').option('port',9999).load()

# TRANSFORM (เหมือน Batch DataFrame!)
result = df.filter(...).groupBy(...).agg(...)
result = df.withWatermark('ts','10m').groupBy(F.window('ts','1m'),'col').count()

# WRITE STREAM
q = result.writeStream \
    .outputMode('append' | 'complete' | 'update') \
    .format('console' | 'parquet' | 'kafka') \
    .option('checkpointLocation','/tmp/ckpt') \
    .trigger(processingTime='10 seconds') \
    .start()
q.awaitTermination(60)
q.stop()

# FOREACH BATCH (custom logic)
def process(batch_df, batch_id):
    batch_df.write.parquet(f'/output/batch_{batch_id}')
result.writeStream.foreachBatch(process).start()
```

## สัปดาห์หน้า (Week 8)
> **EDA & KPI Dashboard** — นำข้อมูล Parquet จาก Streaming มาทำ Exploratory Data Analysis, สร้าง KPI และ Dashboard

---
*References: [Spark Docs](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) | [Kafka Docs](https://kafka.apache.org/documentation/) | [GitHub: polomarcus](https://github.com/polomarcus/Spark-Structured-Streaming-Examples) | [Medium: Real-time Streaming](https://medium.com/@yuvrender.karan/real-time-data-streaming-with-apache-kafka-and-spark-structured-streaming-edb2c47a36f7)*
